# Classification Metrics

**Project question:** How should a decision threshold reflect false-positive and false-negative costs?

By the end of this notebook, you should be able to:

- distinguish probability quality from thresholded class decisions
- select a threshold on validation data using an explicit cost rule
- report confusion-matrix metrics and ROC-AUC once on test data

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

In [ ]:
df = pd.read_csv(DATA / 'evaluation_classification.csv')
X = df.drop(columns=['id', 'event'])
y = df['event']
X_development, X_test, y_development, y_test = train_test_split(
    X, y, stratify=y, test_size=0.20, random_state=4031
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_development, y_development, stratify=y_development, test_size=0.25, random_state=4031
)
pre = ColumnTransformer([
    ('num', StandardScaler(), ['x1', 'x2', 'x3']),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['group']),
])
model = make_pipeline(pre, LogisticRegression(max_iter=1000, random_state=4031))
model.fit(X_train, y_train)
valid_prob = model.predict_proba(X_valid)[:, 1]
y.mean(), y_valid.mean()

We treat a false negative as five times as costly as a false positive for this demonstration. That ratio must come from the real decision context, not from whichever ratio makes the model look best.

In [ ]:
def threshold_metrics(actual, probability, threshold):
    predicted = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    return {
        'threshold': threshold, 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
        'precision': precision, 'recall': recall, 'specificity': specificity,
        'weighted_cost': fp + 5 * fn,
    }

thresholds = np.arange(0.20, 0.81, 0.10)
validation_table = pd.DataFrame(
    threshold_metrics(y_valid, valid_prob, threshold) for threshold in thresholds
)
validation_table

In [ ]:
chosen_threshold = float(
    validation_table.loc[validation_table['weighted_cost'].idxmin(), 'threshold']
)
final_model = make_pipeline(pre, LogisticRegression(max_iter=1000, random_state=4031))
final_model.fit(X_development, y_development)
test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = pd.DataFrame([threshold_metrics(y_test, test_prob, chosen_threshold)])
test_metrics['roc_auc'] = roc_auc_score(y_test, test_prob)
test_metrics

In [ ]:
RocCurveDisplay.from_predictions(y_test, test_prob)
plt.title('Final test ROC curve for the simulated classifier')

**Interpretation:** The threshold was selected without looking at test labels. ROC-AUC evaluates probability ranking across thresholds; it does not encode the stated 5:1 decision cost and does not guarantee calibration.